# Grading INFO 4617 from Canvas

This notebook grades the course's recurring work by reading from and posting to
Canvas: **attendance**, **Textbook Revisions** (weeks 2–9), the **portfolio
checkpoints** (weeks 11–14), and **Notebook Labs**. The rules are in
[`docs/grading.md`](../docs/grading.md); the code is in `grading_lib.py`, next
to this notebook.

Each section works the same way:

1. **Propose.** Read submissions from Canvas and build a table of proposed
   scores and comments. The automated checks fill in what they can measure,
   and set `review` on every row a person has to look at.
2. **Review.** The table is saved as a CSV in `GRADING_DIR`. Open it, change any
   score or comment, and set `review` to `False` once a row is settled.
3. **Post.** With `APPLY = True`, `post()` sends the rows to Canvas. It refuses
   while any row still needs review, and asks you to type how many rows you're
   posting. It then reads every grade and comment back and lists any mismatch.

**Before you commit this notebook, clear its outputs**: they show students'
names and scores, and this repository is public. Student data lives only in
`GRADING_DIR`, outside the repository.

## Setup

Set `CANVAS_TOKEN` in your environment (Canvas → Account → Settings → New
Access Token). `GITHUB_TOKEN` is optional: without it GitHub allows 60 requests
an hour, which isn't enough for a week of Textbook Revisions. A read-only
fine-grained token with no repository access is enough for public repos.

Leave `APPLY = False` until you have reviewed a table and want to post it.

In [ ]:
import os
import pandas as pd
import grading_lib as gl

APPLY = False          # True only when you are ready to post

pd.set_option("display.max_colwidth", 120)
canvas = gl.Canvas(apply=APPLY)
github = gl.GitHub()
print("Student data goes to", gl.GRADING_DIR)
print("Posting is", "ON" if APPLY else "off (dry run)")

## What's left to grade

Submissions in published assignments with no score, or resubmitted after they
were graded. Run this first to see where things stand.

In [ ]:
gl.needs_grading(canvas)

## Attendance

The sign-in sheet is a CSV with one row per sign-in: the student's
**identikey** and a **timestamp** (Mountain time unless the timestamp says
otherwise). Change the column names below if your export uses others.

A week's score is **one point per session held**. A session is a scheduled
meeting day (`course_spec.meeting_days`) that has sign-ins, so a cancelled
class drops out. Sign-ins on any other day are listed and ignored; pass a
make-up session's date in `extra_dates` to count it.

The assignment's points should equal the sessions held. `set_attendance_points`
changes them, but only while the week has no grades.

In [ ]:
SIGNINS = gl.GRADING_DIR / "attendance.csv"
WEEK = 11

signins = gl.load_signins(SIGNINS, id_col="identikey", time_col="timestamp")
attendance, info = gl.propose_attendance(canvas, signins, WEEK, extra_dates=())
info

In [ ]:
# Run once per week, before grading, if points_now differs from sessions_held.
if APPLY and info["points_now"] != info["sessions_held"]:
    gl.set_attendance_points(canvas, WEEK, info["sessions_held"])

In [ ]:
path = gl.save(attendance, f"attendance-w{WEEK:02d}")
print("Review and edit:", path)
gl.summary(attendance)[0]

In [ ]:
# After reviewing the CSV. post() tells you the confirm text to use.
if APPLY:
    print(gl.post(canvas, gl.load(f"attendance-w{WEEK:02d}"), confirm=None))

## Textbook Revisions, weeks 2–9

One point for any issue, pull request, or comment the student made on GitHub
in a class repository. A commit, a compare page, or a pull request inside
their own fork counts too. The checks:

- **GitHub account.** Each student's account is the usual author of their
  links. The mapping table shows any second account (`other_logins`). Confirm
  it by adding it to `LOGINS` and running again.
- **Someone else's item.** If the student commented on it, they get credit
  plus the reminder to link their own comment. Otherwise the row is 0 and
  flagged.
- **The same item in two weeks** gets credit once, in the week it was made.
- **Short comments, non-GitHub links, and other repositories** are flagged or
  noted.

In [ ]:
LOGINS = {}   # Canvas user_id -> confirmed GitHub login(s), e.g. {123456: ["main-login", "second-login"]}

revisions, mapping = gl.propose_revisions(canvas, github, weeks=range(2, 10), logins=LOGINS)
mapping[mapping["other_logins"] != ""]

In [ ]:
path = gl.save(revisions, "revisions-w02-09")
print("Review and edit:", path)
counts, to_review = gl.summary(revisions)
to_review

In [ ]:
if APPLY:
    print(gl.post(canvas, gl.load("revisions-w02-09"), confirm=None))

## Portfolio checkpoint, weeks 11–14

Each student submits the URL of their public project repository. The notebook
clones it into `GRADING_DIR/repos`, checks out its last commit before the
deadline, and checks the week's two criteria (1 point each). The checks look
for the files and README sections the week asks for, so they are a first
pass: any row under 2 points is flagged for you to look at the repo.

In [ ]:
WEEK = 11

portfolio = gl.propose_portfolio(canvas, WEEK)
path = gl.save(portfolio, f"portfolio-w{WEEK:02d}")
print("Review and edit:", path)
gl.summary(portfolio)[1]

In [ ]:
if APPLY:
    print(gl.post(canvas, gl.load(f"portfolio-w{WEEK:02d}"), confirm=None))

## Notebook Labs

Students upload the companion notebook as HTML. The notebook downloads the
uploads, finds the Recommended Exercises, and compares each step with the
companion notebook as it stood before the due date.

- **Rubric.** Criteria 1–4 come from the checks: a valid file, the steps
  attempted, cells run, and the interpretation's length. Criterion 5, whether
  each step does its task, starts at 2. Only a reader can judge it, so lower it
  where needed.
- **Optional steps.** `gl.OPTIONAL_STEPS` lists the steps the instructor told
  students to skip.
- **Blank scaffolds** score 0, with the reminder comment.
- **Late work.** After 24 hours' grace, 1 point is deducted for every started
  day, at most 5. The late comment is added.
- **Other comments** are added for unrun code and for tracebacks. Results that
  no output shows add a comment and a review flag.
- **Flagged for review:** a step missing, a thin interpretation, two files, an
  unusual file, or phrases that look AI-generated. A file the checks can't
  read gets no score and must be scored by hand.

From week 11 the rows carry rubric points, and `post()` fills in the Canvas
rubric. Before week 11, unrun code got full credit; `unrun_full_credit`
switches that by week.

In [ ]:
WEEK = 11

labs = gl.propose_labs(canvas, WEEK, gh=github)
path = gl.save(labs, f"labs-w{WEEK:02d}")
print("Review and edit:", path)
gl.summary(labs)[1]

In [ ]:
if APPLY:
    print(gl.post(canvas, gl.load(f"labs-w{WEEK:02d}"), confirm=None))

## Commendations

A short comment on particularly strong work, at most two or three per
student across the semester, and only for work whose code and output are in
the submission. Comments only; no grade changes.

In [ ]:
commendations = pd.DataFrame([
    # gl.proposal(kind="lab", week=11, assignment_id=2852043, user_id=..., name="...",
    #             action="comment_only", comment="Great find that ..."),
], columns=gl.COLUMNS)
if len(commendations):
    gl.save(commendations, "commendations")
if APPLY and len(commendations):
    print(gl.post(canvas, gl.load("commendations"), confirm=None))

## Check a posted table again

`verify` reads a table's rows back from Canvas and lists any whose grade or
comment doesn't match. `post` runs it automatically; run it again any time.

In [ ]:
# gl.verify(canvas, gl.load("labs-w11"))